# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

#### **One row = one content page (`content_hash_id`) on one day (`report_date`) for one client (`client_hash_id`)**
# 
# ### Time Window for Development:
# - **Feature window:** 2026-01-01 to 2026-03-31 (90 days)
# - **Target window:** 2026-04-01 to 2026-04-30 (30 days after feature window)
# - **Decision point:** 2026-03-31 (end of feature window)
# 
> ### Why this window:
> - 90-day feature window captures trends, seasonality, and sufficient history
> - 30-day target window is long enough to observe meaningful decline but short enough to be actionable
> - The gap between feature window and target window ensures no leakage
> 
> ### Mid-panel development month:
> - **2026-03** for query verification and feature prototyping
> - **Final sealed test month:** 2026-06 (use only for final evaluation)
> 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

> | Feature | Source | Why it matters | Knowable at decision moment because |
> |---------|--------|----------------|-------------------------------------|
> | `avg_impressions_90d` | `fact_content_daily_performance` | High-impression pages matter more when declining | Calculated from 90 days prior to decision point |
> | `position_slope` | `fact_content_daily_performance` | Losing position is a leading indicator of decline | Calculated from last 30 days, fully available at decision point |
> | `ctr_gap` | `fact_content_daily_performance` | Low CTR vs expected at position signals underperformance | Based on historical CTR by position tier, available at decision point |
> | `content_age_days` | `dim_content` | Older pages are more likely to be stale | Content creation date is fixed metadata |
> | `engagement_rate` | `fact_content_daily_performance` | Low engagement signals users find content less useful | Calculated from 90 days prior to decision point |
> 
> ---
> 
> ### LABEL (Target - from future window)
> 
> | Field | Definition | Source | How it's calculated |
> |-------|------------|--------|---------------------|
> | `is_declining_future` | Binary: Did page decline ≥20% in next 30 days? | `fact_content_daily_performance` (future window) | (Impressions next 30 days) / (Impressions previous 30 days) - 1 ≤ -0.20 |
> 
> ---
> 
> ### CONTEXT (Not features, but useful for grouping)
> 
> | Field | Purpose |
> |-------|---------|
> | `client_hash_id` | For client-holdout validation |
> | `content_hash_id` | For identifying the page |
> | `report_date` | For time-based splitting |
> 
> ---
> 
> ### EXCLUDED (Deliberately not used)
> 
> | Field | Why excluded |
> |-------|--------------|
> | `trend_direction` | Leaky — computed from the same window as the label. This is THE trap from the assignment. |
> | `trend_pct` | Leaky — same as above. Derived from the same window. |
> | `health_score` | Not in the warehouse data (product decision, intentionally excluded) |
> | `priority_score` | Not in the warehouse data (product decision, intentionally excluded) |
> | `action_type` | Not in the warehouse data (product decision, intentionally excluded) |
> | Raw client names | Scrambled before release, never available |
> | Raw URLs | Scrambled before release, never available |
> | Raw queries | Scrambled before release, never available |
> 
> ---


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# pip install duckdb

In [9]:
# Import libraries
import pandas as pd
import numpy as np
import duckdb
import os
from datetime import datetime, timedelta

print("=" * 60)
print("SETUP: Loading Data")
print("=" * 60)

# Check if we have HF_TOKEN
if 'HF_TOKEN' in os.environ:
    print(f"HF_TOKEN found (length: {len(os.environ['HF_TOKEN'])})")
    USE_WAREHOUSE = True
else:
    print("HF_TOKEN not found. Using starter dataset instead.")
    USE_WAREHOUSE = False

# Load the data
if USE_WAREHOUSE:
    # Try to load from Hugging Face warehouse
    try:
        # Query March 2026 data from warehouse
        query = """
        SELECT 
            report_date,
            client_hash_id,
            content_hash_id,
            impressions,
            clicks,
            avg_position,
            sessions,
            engaged_sessions,
            scroll_events,
            engagement_rate,
            scroll_rate
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*.parquet'
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        """
        
        df = duckdb.query(query).to_df()
        print(f"Loaded {len(df):,} rows from warehouse (March 2026)")
        
        # Also load dim_content for content metadata
        query_dim = """
        SELECT 
            content_hash_id,
            client_hash_id,
            content_created_at,
            content_type,
            word_count,
            char_count
        FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content/*.parquet'
        LIMIT 10000
        """
        
        df_dim = duckdb.query(query_dim).to_df()
        print(f"Loaded {len(df_dim):,} rows from dim_content")
        
    except Exception as e:
        print(f"Could not query warehouse: {e}")
        print("Falling back to starter dataset...")
        df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
        df_dim = None
        print(f"Loaded starter dataset: {len(df):,} rows")
else:
    # Fallback to starter dataset
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
    df_dim = None
    print(f"Loaded starter dataset: {len(df):,} rows")

# Show what we have
print(f"\nColumns in dataset:")
print(df.columns.tolist())

SETUP: Loading Data
HF_TOKEN not found. Using starter dataset instead.
Loaded starter dataset: 30,000 rows

Columns in dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


### Query 1: Verify the grain

**Claim:** One row = one content page on one day

In [10]:
print("=" * 60)
print("QUERY 1: VERIFY THE GRAIN")
print("=" * 60)

if 'report_date' in df.columns:
    # Warehouse data - check daily grain
    grain_check = df.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size()
    duplicates = grain_check[grain_check > 1].shape[0]
    
    print(f"Total rows in sample: {len(df):,}")
    print(f"Unique (date, client, content) combinations: {len(grain_check):,}")
    print(f"Duplicate combinations: {duplicates}")
    
    if duplicates == 0:
        print("GRAIN VERIFIED: Each row is one (date, client, content) combination.")
    else:
        print(f"Found {duplicates} duplicate combinations.")
    
    # Show a sample row
    print(f"\nSample row:")
    print(df[['report_date', 'client_hash_id', 'content_hash_id', 'impressions', 'clicks']].head(1))
    
else:
    # Starter dataset - aggregated by page
    grain_check = df.groupby(['content_id', 'client_id']).size()
    print(f"Total rows: {len(df):,}")
    print(f"Unique (content, client) combinations: {len(grain_check):,}")
    print("For starter data, each row = one content page (aggregated)")
    
    # Show a sample row
    print(f"\nSample row:")
    print(df[['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'trend_direction']].head(1))

QUERY 1: VERIFY THE GRAIN
Total rows: 30,000
Unique (content, client) combinations: 30,000
For starter data, each row = one content page (aggregated)

Sample row:
             content_id          client_id  impressions_90d  clicks_90d  \
0  content_304f48230142  client_f369cb89fc             3803          29   

  trend_direction  
0            down  


### Query 2: Row count, date span, and distribution

In [11]:
print("=" * 60)
print("QUERY 2: ROW COUNT AND DATE SPAN")
print("=" * 60)

if 'report_date' in df.columns:
    # Warehouse data
    df['report_date'] = pd.to_datetime(df['report_date'])
    
    print(f"Date range: {df['report_date'].min()} to {df['report_date'].max()}")
    print(f"Total rows in sample: {len(df):,}")
    print(f"Unique clients: {df['client_hash_id'].nunique():,}")
    print(f"Unique content items: {df['content_hash_id'].nunique():,}")
    
    # Daily distribution
    rows_by_date = df.groupby('report_date').size()
    print(f"\nDaily row count distribution:")
    print(f"  Mean: {rows_by_date.mean():.0f}")
    print(f"  Min: {rows_by_date.min():.0f}")
    print(f"  Max: {rows_by_date.max():.0f}")
    print(f"  Days: {len(rows_by_date)}")
    
    # Show first 5 days
    print(f"\nFirst 5 days:")
    print(rows_by_date.head(5))
    
    print("\nData is at daily grain with ~30 days of history.")
    
else:
    # Starter dataset
    print(f"Date range: Not applicable (aggregated starter data)")
    print(f"Total rows: {len(df):,}")
    print(f"Unique clients: {df['client_id'].nunique():,}")
    print(f"Unique content items: {df['content_id'].nunique():,}")
    
    # Check distribution of trend_direction
    print(f"\nTarget distribution:")
    if 'trend_direction' in df.columns:
        print(df['trend_direction'].value_counts())
        declining = df[df['trend_direction'] == 'down'].shape[0]
        print(f"Declining: {declining:,} ({declining/len(df)*100:.1f}%)")
    
    print("\nStarter data has 30,000 aggregated pages across multiple clients.")

QUERY 2: ROW COUNT AND DATE SPAN
Date range: Not applicable (aggregated starter data)
Total rows: 30,000
Unique clients: 32
Unique content items: 30,000

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
Declining: 16,262 (54.2%)

Starter data has 30,000 aggregated pages across multiple clients.


### Query 3: Missing values and availability

In [12]:
print("=" * 60)
print("QUERY 3: MISSING VALUES AND AVAILABILITY")
print("=" * 60)

if 'report_date' in df.columns:
    # Warehouse data - check key signals
    total = len(df)
    print(f"Total rows: {total:,}")
    print("\nSignal availability:")
    
    # Check each key column
    for col in ['impressions', 'clicks', 'sessions', 'avg_position']:
        if col in df.columns:
            available = df[col].notna().sum()
            print(f"  {col}: {available:,} ({available/total*100:.1f}%)")
    
    # Check rows with all key signals
    if 'impressions' in df.columns and 'clicks' in df.columns and 'sessions' in df.columns:
        has_all = df[
            df['impressions'].notna() & 
            df['clicks'].notna() & 
            df['sessions'].notna()
        ].shape[0]
        print(f"\nRows with all three signals: {has_all:,} ({has_all/total*100:.1f}%)")
        
        # Filter recommendation
        print(f"\nFor my lane, I'll filter to pages with >=500 impressions and sessions > 0")
        
else:
    # Starter dataset
    total = len(df)
    print(f"Total rows: {total:,}")
    print("\nSignal availability:")
    
    for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr']:
        if col in df.columns:
            available = df[col].notna().sum()
            print(f"  {col}: {available:,} ({available/total*100:.1f}%)")
    
    # Check high-demand pages
    if 'impressions_90d' in df.columns:
        high_demand = df[df['impressions_90d'] >= 500].shape[0]
        print(f"\nRows with >=500 impressions: {high_demand:,} ({high_demand/total*100:.1f}%)")
        print(f"Filtering to >=500 impressions removes noise but keeps {high_demand} pages.")

QUERY 3: MISSING VALUES AND AVAILABILITY
Total rows: 30,000

Signal availability:
  impressions_90d: 30,000 (100.0%)
  clicks_90d: 30,000 (100.0%)
  sessions_90d: 30,000 (100.0%)
  ctr: 30,000 (100.0%)

Rows with >=500 impressions: 16,726 (55.8%)
Filtering to >=500 impressions removes noise but keeps 16726 pages.


### Query 4: Verify the 90-day feature window and 30-day target window (simulated)

In [13]:
print("=" * 60)
print("QUERY 4: WINDOW VERIFICATION (SIMULATED)")
print("=" * 60)

# Since we only have aggregated starter data or a 30-day slice,
# we'll simulate what the window verification would look like

print("For the actual warehouse data:")

# Feature window
print(f"\nFeature window (90 days):")
print(f"  Start: 2026-01-01")
print(f"  End: 2026-03-31")
print(f"  Available signals: impressions, clicks, position, sessions, engagement, scroll")

print(f"\nTarget window (30 days):")
print(f"  Start: 2026-04-01")
print(f"  End: 2026-04-30")
print(f"  Target: is_declining_future (impressions drop >= 20%)")

print(f"\nDecision point: 2026-03-31")
print(f"  Features: calculate aggregated signals from 2026-01-01 to 2026-03-31")
print(f"  Label: calculate from 2026-04-01 to 2026-04-30")

print(f"\nNo overlap between feature window and target window - NO LEAKAGE")

print("\n" + "=" * 60)
print("SAMPLE RECORD (What one row would look like in the feature frame)")
print("=" * 60)

# Create a sample feature frame
sample_record = {
    'content_hash_id': 'abc123_hash',
    'client_hash_id': 'xyz789_hash',
    'decision_date': '2026-03-31',
    'avg_impressions_90d': 1250,
    'position_slope': -0.15,
    'ctr_gap': -0.35,
    'content_age_days': 210,
    'engagement_rate': 0.45,
    'is_declining_future': 1
}

sample_df = pd.DataFrame([sample_record])
print(sample_df.T)

QUERY 4: WINDOW VERIFICATION (SIMULATED)
For the actual warehouse data:

Feature window (90 days):
  Start: 2026-01-01
  End: 2026-03-31
  Available signals: impressions, clicks, position, sessions, engagement, scroll

Target window (30 days):
  Start: 2026-04-01
  End: 2026-04-30
  Target: is_declining_future (impressions drop >= 20%)

Decision point: 2026-03-31
  Features: calculate aggregated signals from 2026-01-01 to 2026-03-31
  Label: calculate from 2026-04-01 to 2026-04-30

No overlap between feature window and target window - NO LEAKAGE

SAMPLE RECORD (What one row would look like in the feature frame)
                               0
content_hash_id      abc123_hash
client_hash_id       xyz789_hash
decision_date         2026-03-31
avg_impressions_90d         1250
position_slope             -0.15
ctr_gap                    -0.35
content_age_days             210
engagement_rate             0.45
is_declining_future            1


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

> | Limit | Why it matters | What I can't claim |
> |-------|----------------|-------------------|
> | **Correlation ≠ Causation** | The data shows patterns, but doesn't prove what causes them | "I proved that [feature] causes decline" |
> | **Unbalanced history** | Different clients have different start dates (9 clients have 12+ months, others less) | "This model works equally well for all clients" |
> | **GSC-only early rows** | `ga4_data_available = FALSE` for early periods means some signals are missing | "These findings apply to all time periods" |
> | **No semantic understanding** | Only structured signals, no article text | "The model 'understands' content meaning" |
> | **Observational, not experimental** | We observed what happened, we didn't run experiments | "Refreshing these pages WILL cause recovery" |
> | **Proxy target** | `trend_direction` is from current window, not future | "The model predicts future decline" (until I build the future-looking label) |
> | **Limited to observable signals** | No product decisions, no health_score, no action_type | "I can build an automated decision system" |
> 
> ### What I CAN claim:
> 
> | Safe claim | Why it's safe |
> |------------|---------------|
> | "I can rank pages by likelihood of decline based on observable historical signals" | Features are from prior window, label is from future window |
> | "This ranking outperforms a fixed heuristic baseline on the starter dataset" | Measured and verified with Precision@50 |
> | "This is a decision-support tool, not an automation" | Output is a ranked queue for human review |
> 
> ---


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.